# Install the Package
Here we're installing it directly from GitHub while it's in development.

In [ ]:
!pip install 'r2-db2[flask,anthropic]'

# Download a Sample Database

In [ ]:
import httpx

with open("Chinook.sqlite", "wb") as f:
    with httpx.stream("GET", "https://r2-db2.ai/Chinook.sqlite") as response:
        for chunk in response.iter_bytes():
            f.write(chunk)

# Imports

In [ ]:
from r2-db2 import Agent, AgentConfig
from r2-db2.servers.fastapi import R2-DB2FastAPIServer
from r2-db2.core.registry import ToolRegistry
from r2-db2.core.user import UserResolver, User, RequestContext
from r2-db2.integrations.anthropic import AnthropicLlmService
from r2-db2.tools import RunSqlTool, VisualizeDataTool
from r2-db2.integrations.sqlite import SqliteRunner
from r2-db2.tools.agent_memory import SaveQuestionToolArgsTool, SearchSavedCorrectToolUsesTool
from r2-db2.integrations.local.agent_memory import DemoAgentMemory
from r2-db2.capabilities.sql_runner import RunSqlToolArgs
from r2-db2.tools.visualize_data import VisualizeDataArgs

# Define your User Authentication
Here we're going to say that if you're logged in as `admin@example.com` then you're in the `admin` group, otherwise you're in the `user` group

In [ ]:
class SimpleUserResolver(UserResolver):
    async def resolve_user(self, request_context: RequestContext) -> User:
        # In production, validate cookies/JWTs here
        user_email = request_context.get_cookie('r2_db2_email')
        if not user_email:
            raise ValueError("Missing 'r2_db2_email' cookie for user identification")
        
        print(f"Resolving user for email: {user_email}")

        if user_email == "admin@example.com":
            return User(id="admin1", email=user_email, group_memberships=['admin'])
        
        return User(id="user1", email=user_email, group_memberships=['user'])

# Define the Tools and Access Control

In [ ]:
tools = ToolRegistry()
tools.register_local_tool(RunSqlTool(sql_runner=SqliteRunner(database_path="./Chinook.sqlite")), access_groups=['admin', 'user'])
tools.register_local_tool(VisualizeDataTool(), access_groups=['admin', 'user'])
agent_memory = DemoAgentMemory(max_items=1000)
tools.register_local_tool(SaveQuestionToolArgsTool(), access_groups=['admin'])
tools.register_local_tool(SearchSavedCorrectToolUsesTool(), access_groups=['admin', 'user'])

In [ ]:
# Set up LLM
llm = AnthropicLlmService(model="claude-sonnet-4-5", api_key="sk-ant-...")

# Create agent with your options
agent = Agent(
    llm_service=llm,
    tool_registry=tools,
    user_resolver=SimpleUserResolver(),
    config=AgentConfig(),
    agent_memory=agent_memory
)

# 4. Create and run server
server = R2-DB2FastAPIServer(agent)
server.run()